In [30]:
import os
from getpass import getpass
os.environ['LANGSMITH_TRACING'] = "true"
os.environ['LANGSMITH_API_KEY'] = getpass("Langsmith API Key: ")
os.environ['LANGSMITH_PROJECT'] = "gemini_api"
os.environ["GOOGLE_API_KEY"] = getpass("Enter your Gemini API key: ")

In [55]:
import langchain_community
import langchain_text_splitters
import chromadb

print("Imports Successful")

Imports Successful


In [ ]:
from langchain_classic import hub
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# Initialize the Gemini Flash model
llm = ChatGoogleGenerativeAI(
    #model="gemini-3.8-flash",  # listed in Google's current model catalog
    model="gemini-3.5-flash-lite",
    temperature=0.3,
    #timeout=120,
    #max_retries=0,
    #max_tokens=128,
)

#Test the model
#print("Sending request...", flush=True)
#response = llm.invoke("Reply with exactly: Hello")
#print(response.content)




Sending request...


c:\Users\ssury\Desktop\Programming\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'Hello', 'extras': {'signature': 'EmAKXgFpFH0TAs+InRQOGNvPhE8vmVuZ0SdTWuNLYMKpEpJlkHqtj7nHe4Dn2cCNFmOhoircnWncO+Ul6+Ghvz3VWZ6GbRv16H6LmHTC3GVQ9kubhM0AgHXgaCvhw3mYAjE='}}]


In [68]:
# Documents
question = "What kinds of pets do I like?"
document = "My favorite pet is a dog."

Count tokens considering -> 4 char / token

In [69]:
import tiktoken

def num_tokens_from_string(string: str , encoding_name: str) -> int:
    """Returns no.of tokens in a text string"""
    encoding = tiktoken.get_encoding(encoding_name)
    num_tokens = len(encoding.encode(string))
    return num_tokens

num_tokens_from_string(document, "cl100k_base")

7

Text Embedding Models

In [70]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# Initialize the embedding model recommended for Gemini RAG tasks
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2"
)

# Test embedding a query string
#vector = embeddings.embed_query("What is retrieval-augmented generation?")
#print(f"Embedding vector length: {len(vector)}")

query_result = embeddings.embed_query(question)
document_result = embeddings.embed_query(document)
print(len(query_result),len(document_result))

3072 3072


Cosine Similarity

In [71]:
import numpy as np

def cosine_similarity(vec1,vec2):
    dot_product = np.dot(vec1,vec2)
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)
    return dot_product / (norm_vec1 * norm_vec2)

similarity = cosine_similarity(query_result , document_result)
print("Cosine Similarity: " , similarity)

Cosine Similarity:  0.8078864721468739


Document Loaders

In [72]:
### INDEXING ###

# Load Blog

import bs4
from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
blog_docs = loader.load()

Splitter - This text splitter is the recommended one for generic text. It is parameterized by a list of characters. It tries to split on them in order until the chunks are small enough. The default list is ["\n\n", "\n", " ", ""]. This has the effect of trying to keep all paragraphs (and then sentences, and then words) together as long as possible, as those would generically seem to be the strongest semantically related pieces of text.

In [73]:
#Split
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size = 300,
    chunk_overlap = 50
)

splits = text_splitter.split_documents(blog_docs)

Retrieval

VectorStores

In [74]:
#Index

from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma
vectorstore = Chroma.from_documents(documents = splits , embedding = embeddings )

retriever = vectorstore.as_retriever(search_kwargs = {"k" : 1})



In [75]:
docs = retriever.invoke("What is Task Decompostition? , Generate me 10 other relevant questions from the document along with their answers")
for doc in docs:
    print(doc.page_content)

Component One: Planning#
A complicated task usually involves many steps. An agent needs to know what they are and plan ahead.
Task Decomposition#
Chain of thought (CoT; Wei et al. 2022) has become a standard prompting technique for enhancing model performance on complex tasks. The model is instructed to “think step by step” to utilize more test-time computation to decompose hard tasks into smaller and simpler steps. CoT transforms big tasks into multiple manageable tasks and shed lights into an interpretation of the model’s thinking process.
Tree of Thoughts (Yao et al. 2023) extends CoT by exploring multiple reasoning possibilities at each step. It first decomposes the problem into multiple thought steps and generates multiple thoughts per step, creating a tree structure. The search process can be BFS (breadth-first search) or DFS (depth-first search) with each state evaluated by a classifier (via a prompt) or majority vote.
Task decomposition can be done (1) by LLM with simple prompt

Generation

In [76]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

#Prompt

template = """Answer the question based only on the following context:
{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

#chain

chain = prompt | llm

#Run

chain.invoke({"context":docs,"question":"What is Task Decomposition? Generate me 10 other relevant questions from the document along with their answers"})

c:\Users\ssury\Desktop\Programming\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


AIMessage(content=[{'type': 'text', 'text': 'Based on the provided context, **Task Decomposition** is a component of planning where a complicated task involving many steps is broken down into smaller and simpler steps. This can be achieved using techniques like Chain of Thought or Tree of Thoughts, and can be done (1) by LLM with simple prompting, (2) by using task-specific instructions, or (3) with human inputs.\n\n***\n\n### 10 Other Relevant Questions and Answers\n\n**1. Q: What is the purpose of Component One: Planning for an agent?**\nA: A complicated task usually involves many steps, and an agent needs to know what those steps are and plan ahead.\n\n**2. Q: What is Chain of Thought (CoT)?**\nA: CoT is a standard prompting technique (Wei et al. 2022) used to enhance model performance on complex tasks by instructing the model to "think step by step" to utilize more test-time computation.\n\n**3. Q: How does Chain of Thought help with complex tasks?**\nA: It transforms big tasks int

RAG Chains

In [77]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    {"context" : retriever , "question" : RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

rag_chain.invoke("What is task decompostion? Generate me 10 other relevant questions from the document along with their answers")

c:\Users\ssury\Desktop\Programming\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


'Based on the provided context, **task decomposition** is a process where a complicated or hard task is broken down into smaller, simpler, and more manageable steps or subgoals (such as through techniques like Chain of Thought). \n\nHere are 10 other relevant questions from the document along with their answers:\n\n**1. Question:** What is the purpose of Component One in an agent system?\n**Answer:** It is for planning, as a complicated task usually involves many steps and an agent needs to know what they are and plan ahead.\n\n**2. Question:** What does Chain of Thought (CoT) instruct the model to do?\n**Answer:** The model is instructed to “think step by step” to utilize more test-time computation.\n\n**3. Question:** What are the two main benefits of the Chain of Thought (CoT) technique?\n**Answer:** It transforms big tasks into multiple manageable tasks and sheds light into an interpretation of the model’s thinking process.\n\n**4. Question:** How does the Tree of Thoughts (ToT) ex